# MORGAN fingerprints with Data 4.3

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('data_4.csv')
df_cdd = df_cdd.rename(columns={
    "GU CRC L6: Average EC50": "gu_ec50",
    "GU CRC L6: Average EC50 Standard Deviation (±)": "gu_std",
    "GU CRC L6: Average EC50 Count": "gu_count",
    "GU CRC L6: Average Emax": "gu_emax",
    "B2 cAMP CRC L6: Average Emax Standard Deviation (±)": "gu_emax_std",
    "B2 cAMP CRC L6: Average Emax Count": "gu_emax_count",
    "B2 cAMP CRC L6: Average EC50 2": "camp_ec50",
    'B2 cAMP CRC L6: Average EC50 2 Standard Deviation (±)': "camp_std",
    'B2 cAMP CRC L6: Average EC50 2 Count': "camp_count",
    "B2 cAMP CRC L6: Average Emax": "camp_emax",
    "GU CRC L6: Average Emax Standard Deviation (±)": "camp_emax_std",
    "GU CRC L6: Average Emax Count": "camp_emax_count"
})

Calculates the maximum value from the CRC doses for glucose uptake and cAMP

In [ ]:
camp_cols = [
    "B2 cAMP CRC L6: cAMP (-5) (of iso)",
    "B2 cAMP CRC L6: cAMP (-6) (of iso)",
    "B2 cAMP CRC L6: cAMP (-7) (of iso)",
    "B2 cAMP CRC L6: cAMP (-8) (of iso)",
    "B2 cAMP CRC L6: cAMP (-9) (of iso)",
    "B2 cAMP CRC L6: cAMP (-10) (of iso)"
]

gu_cols = [
    "GU CRC L6: GU (-5) (of iso)",
    "GU CRC L6: GU (-6) (of iso)",
    "GU CRC L6: GU (-7) (of iso)",
    "GU CRC L6: GU (-8) (of iso)",
    "GU CRC L6: GU (-9) (of iso)",
    "GU CRC L6: GU (-10) (of iso)"
]

df_cdd["camp_emax"] = df_cdd[camp_cols].max(axis=1, skipna=True)
df_cdd["gu_emax"] = df_cdd[gu_cols].max(axis=1, skipna=True)

Drops the columns that will not be needed for the ML models and Explainable methods

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'B2 cAMP CRC L6: Basal (of iso)',
                      'B2 cAMP CRC L6: cAMP Clen (of iso)',
                      "B2 cAMP CRC L6: cAMP (-5) (of iso)",
                      'B2 cAMP CRC L6: cAMP (-6) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-7) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-8) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-9) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-10) (of iso)',
                      'B2 cAMP CRC L6: Top (of iso)',
                      'B2 cAMP CRC L6: Bottom (of iso)',
                      'B2 cAMP CRC L6: Average EC50 batch',
                      'B2 cAMP CRC L6: Average EC50 batch Standard Deviation (±)',
                      'B2 cAMP CRC L6: Average EC50 batch Count',
                      'B2 cAMP CRC L6: Emax',
                      'GU CRC L6: GU (-5) (of iso)',
                      'GU CRC L6: GU (-6) (of iso)',
                      'GU CRC L6: GU (-7) (of iso)',
                      'GU CRC L6: GU (-8) (of iso)',
                      'GU CRC L6: GU (-9) (of iso)',
                      'GU CRC L6: GU (-10) (of iso)',
                      'GU CRC L6: Bottom L6G (of iso)',
                      'GU CRC L6: Top L6G (of iso)',
                      'GU CRC L6: Average EC50 batch',
                      'GU CRC L6: Average EC50 batch Standard Deviation (±)',
                      'GU CRC L6: Average EC50 batch Count',
                      'GU CRC L6: Emax'
                    ], axis=1)


Combine replicates of the same molecular compounds, so that only 1 data for each unique molecule remains

In [ ]:
import numpy as np

# Group by Molecule Name and SMILES, then aggregate
df_cdd = df_cdd.groupby(['Molecule Name', 'SMILES']).agg({
    # For EC50 values: take mean
    'camp_ec50': 'mean',
    'gu_ec50': 'mean',
    
    # For Emax values: take mean
    'camp_emax': 'mean',
    'gu_emax': 'mean',
    
    # For standard deviations: pool them using root mean square
    'camp_std': lambda x: np.sqrt(np.mean(x**2)),
    'gu_std': lambda x: np.sqrt(np.mean(x**2)),
    'gu_emax_std': lambda x: np.sqrt(np.mean(x**2)),
    'camp_emax_std': lambda x: np.sqrt(np.mean(x**2)),
}).reset_index()

Drop the standard deviation and count features

In [ ]:
df_cdd = df_cdd.drop(['camp_std',
                      'gu_emax_std',
                      'gu_std',
                      'camp_emax_std'
                    ], axis=1)

df_cdd = df_cdd[["Molecule Name", "SMILES", "gu_ec50", "gu_emax", "camp_ec50", "camp_emax"]]

df_cdd = df_cdd.dropna(subset=[ 'gu_ec50',
                                'camp_ec50' ])

In [ ]:
df = df_cdd
df.columns = df.columns.str.strip()

# Basic sanity check
TARGET_NAMES = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']
required = {'Molecule Name', 'SMILES'} | set(TARGET_NAMES)
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

print(f'Dataset shape: {df.shape}')
print(df[TARGET_NAMES].describe())
df.head()

### Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy

# RDKit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

# Sklearn
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# SHAP
import shap

# PyTorch & PyG
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('All imports successful.')

In [ ]:
# New target variables
TARGET_COLS = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']

# Visual: distributions for the four figures
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

colors = ['steelblue', 'teal', 'darkorange', 'firebrick']

for ax, col, color in zip(axes.flat, TARGET_COLS, colors):
    ax.hist(df_cdd[col], bins=30, color=color, edgecolor='black', alpha=0.85)
    ax.axvline(df_cdd[col].mean(), color='black', linestyle='--', linewidth=1,
               label=f'mean={df_cdd[col].mean():.3f}')
    ax.set_title(f'{col} distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Distributions of EC50 and Emax for GU and CAMP', fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Morgan Fingerprints (2048-bit, radius=2)

> Morgan (ECFP4) fingerprints encode circular substructure environments up to bond-radius 2. Each of the 2048 bits corresponds to a hashed substructure fragment. Zero-variance bits are removed before modelling.

In [ ]:
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048

def smiles_to_morgan(smi, radius=MORGAN_RADIUS, n_bits=MORGAN_NBITS):
    mol = Chem.MolFromSmiles(smi)
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    return np.array(fp)

X = np.vstack(df['SMILES'].apply(smiles_to_morgan).values)  # (N, 2048)
y = df[TARGET_NAMES].values  # (N, 4)

# Morgan bit names for later labelling
morgan_names = [f'Morgan_{i}' for i in range(MORGAN_NBITS)]

# Remove always-zero or always-one bits (no variance)
var_mask = (X.var(axis=0) > 0)
X_filtered = X[:, var_mask]
names_filtered = [morgan_names[i] for i, v in enumerate(var_mask) if v]

print(f'Morgan shape (raw): {X.shape}')
print(f'Morgan shape (after removing zero-variance bits): {X_filtered.shape}')

# Visualise bit frequency
bit_freq = X_filtered.mean(axis=0)
plt.figure(figsize=(14,3))
plt.bar(range(len(bit_freq)), bit_freq, color='steelblue')
plt.xlabel('Morgan bit index (filtered)')
plt.ylabel('Frequency')
plt.title('Morgan Fingerprint Bit Frequencies across the Dataset')
plt.tight_layout()
plt.show()

## 3. Cross-Validation Setup

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_metrics(y_true, y_pred, target_names=TARGET_NAMES):
    """Return per-target RMSE, MAE, R2."""
    results = {}
    for i, name in enumerate(target_names):
        results[name] = {
            'RMSE': np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            'MAE':  mean_absolute_error(y_true[:,i], y_pred[:,i]),
            'R2':   r2_score(y_true[:,i], y_pred[:,i])
        }
    return results

def print_cv_summary(all_fold_metrics, model_name):
    print(f'\n{"─"*65}')
    print(f'  {model_name} — 5-Fold CV Summary')
    print(f'{"─"*65}')
    for target in TARGET_NAMES:
        rmse_vals = [m[target]['RMSE'] for m in all_fold_metrics]
        mae_vals  = [m[target]['MAE']  for m in all_fold_metrics]
        r2_vals   = [m[target]['R2']   for m in all_fold_metrics]
        print(f'  [{target.upper()}]  RMSE: {np.mean(rmse_vals):.4f}±{np.std(rmse_vals):.4f} | '
              f'MAE: {np.mean(mae_vals):.4f}±{np.std(mae_vals):.4f} | '
              f'R2: {np.mean(r2_vals):.4f}±{np.std(r2_vals):.4f}')
    print(f'{"─"*65}')

## 4. Machine Learning models 
### 4.1 Multi-output support vector regressor

In [ ]:
svr_fold_metrics = []
svr_fold_models  = []   # store (scaler, model) per fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)

    # MultiOutputRegressor wraps SVR for 4-target regression
    base_svr = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
    mo_svr   = MultiOutputRegressor(base_svr, n_jobs=-1)
    mo_svr.fit(X_tr_sc, y_tr)

    y_pred = mo_svr.predict(X_val_sc)
    metrics = cv_metrics(y_val, y_pred)
    svr_fold_metrics.append(metrics)
    svr_fold_models.append((scaler, mo_svr, train_idx, val_idx))
    rmse_str = '  '.join([f'{t}_RMSE={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES])
    print(f'Fold {fold+1}: {rmse_str}')

print_cv_summary(svr_fold_metrics, 'Multi-Output SVR')

### 4.2 Multi-output random forest

In [ ]:
rf_fold_metrics = []
rf_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        random_state=SEED, n_jobs=-1
    )
    # RandomForestRegressor natively handles multi-output
    rf.fit(X_tr, y_tr)

    y_pred = rf.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    rf_fold_metrics.append(metrics)
    rf_fold_models.append((rf, train_idx, val_idx))
    rmse_str = '  '.join([f'{t}_RMSE={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES])
    print(f'Fold {fold+1}: {rmse_str}')

print_cv_summary(rf_fold_metrics, 'Multi-Output Random Forest')

### 4.3 Multi-output XGBoost

In [ ]:
xgb_fold_metrics = []
xgb_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # XGBoost >= 2.0 supports multi-output natively with multi_strategy
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        multi_strategy='multi_output_tree',   # native multi-output
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = xgb_model.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    xgb_fold_metrics.append(metrics)
    xgb_fold_models.append((xgb_model, train_idx, val_idx))
    rmse_str = '  '.join([f'{t}_RMSE={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES])
    print(f'Fold {fold+1}: {rmse_str}')

print_cv_summary(xgb_fold_metrics, 'Multi-Output XGBoost')

### 4.4 Model performance comparison

In [ ]:
def aggregate_cv(fold_metrics, model_name):
    rows = []
    for target in ['gu_ec50','gu_emax', 'camp_ec50', 'camp_emax']:
        for metric in ['RMSE','R2']:
            vals = [m[target][metric] for m in fold_metrics]
            rows.append({'Model': model_name, 'Target': target,
                         'Metric': metric, 'Mean': np.mean(vals), 'Std': np.std(vals)})
    return pd.DataFrame(rows)

results_df = pd.concat([
    aggregate_cv(svr_fold_metrics,  'SVR'),
    aggregate_cv(rf_fold_metrics,   'Random Forest'),
    aggregate_cv(xgb_fold_metrics,  'XGBoost'),
], ignore_index=True)

# Pivot for readability
pivot = results_df.pivot_table(index=['Model','Target'], columns='Metric', values=['Mean','Std'])
print(pivot.round(4))

# Bar chart with best model highlighted in green
fig, axes = plt.subplots(2, 2, figsize=(14, 5))

for ax, target in zip(axes, ['gu_ec50','gu_emax', 'camp_ec50', 'camp_emax']):
    sub = results_df[(results_df['Target'] == target) & (results_df['Metric'] == 'RMSE')]

    # Identify best model (lowest RMSE)
    best_idx = sub['Mean'].idxmin()
    best_model = sub.loc[best_idx, 'Model']

    # Assign colors: green for best, blue for others
    colors = ['green' if m == best_model else 'steelblue' for m in sub['Model']]

    ax.bar(sub['Model'], sub['Mean'], yerr=sub['Std'], 
           capsize=5, color=colors, alpha=0.85)

    ax.set_title(f'{target.upper()} — Mean RMSE (5-Fold CV)')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


print('\nExpected output: bar chart + pivot table with RMSE/MAE/R2 for all 4 models on both targets.')

In [ ]:
# ── Cross-model overfit summary heatmap ───────────────────────────────────
# Collects the mean Train RMSE and Val RMSE per model/target across all folds,
# then visualises the generalisation gap.

def compute_gap_summary(fold_models, model_type):
    """
    model_type: 'svr' | 'rf' | 'xgb'
    Returns DataFrame with columns [Model, Target, TrainRMSE, ValRMSE, Gap]
    """
    rows = []
    for fold_idx, entry in enumerate(fold_models):
        if model_type == 'svr':
            scaler, model, tr_idx, val_idx = entry
            X_tr  = scaler.transform(X_filtered[tr_idx])
            X_val = scaler.transform(X_filtered[val_idx])
        else:
            model, tr_idx, val_idx = entry
            X_tr  = X_filtered[tr_idx]
            X_val = X_filtered[val_idx]

        y_tr, y_val   = y[tr_idx], y[val_idx]
        y_pred_tr  = model.predict(X_tr)
        y_pred_val = model.predict(X_val)

        for t_idx, t in enumerate(TARGET_NAMES):
            tr_rmse  = np.sqrt(mean_squared_error(y_tr[:, t_idx],  y_pred_tr[:, t_idx]))
            val_rmse = np.sqrt(mean_squared_error(y_val[:, t_idx], y_pred_val[:, t_idx]))
            rows.append({'Model': model_type.upper(), 'Target': t, 'Fold': fold_idx+1,
                         'TrainRMSE': tr_rmse, 'ValRMSE': val_rmse, 'Gap': val_rmse - tr_rmse})
    return pd.DataFrame(rows)

gap_df = pd.concat([
    compute_gap_summary(svr_fold_models, 'svr'),
    compute_gap_summary(rf_fold_models,  'rf'),
    compute_gap_summary(xgb_fold_models, 'xgb'),
], ignore_index=True)

# Mean across folds
gap_summary = gap_df.groupby(['Model', 'Target'])[['TrainRMSE', 'ValRMSE', 'Gap']].mean().reset_index()

# ── Heatmap of Val–Train gap ───────────────────────────────────────────────
gap_pivot = gap_summary.pivot(index='Model', columns='Target', values='Gap')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# (a) Gap heatmap
sns.heatmap(gap_pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
            center=0, linewidths=0.5, ax=axes[0])
axes[0].set_title('Val − Train RMSE Gap\n(higher = more overfit)', fontsize=11)

# (b) Train RMSE heatmap
train_pivot = gap_summary.pivot(index='Model', columns='Target', values='TrainRMSE')
sns.heatmap(train_pivot, annot=True, fmt='.3f', cmap='Blues',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Mean Train RMSE\n(high = underfitting)', fontsize=11)

# (c) Val RMSE heatmap
val_pivot = gap_summary.pivot(index='Model', columns='Target', values='ValRMSE')
sns.heatmap(val_pivot, annot=True, fmt='.3f', cmap='Oranges',
            linewidths=0.5, ax=axes[2])
axes[2].set_title('Mean Val RMSE\n(high = poor generalisation)', fontsize=11)

plt.suptitle('Overfit / Underfit Diagnostic — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Diagnosis printout ─────────────────────────────────────────────────────
print("\nDiagnosis (mean across folds):")
print(f"{'Model':<6} {'Target':<14} {'TrainRMSE':>12} {'ValRMSE':>10} {'Gap':>8}  {'Status'}")
print("─" * 68)
for _, row in gap_summary.iterrows():
    if row['Gap'] > 0.3:
        status = "OVERFIT"
    elif row['TrainRMSE'] > 0.8:
        status = "UNDERFIT"
    else:
        status = "OK"
    print(f"{row['Model']:<6} {row['Target']:<14} {row['TrainRMSE']:>12.4f} "
          f"{row['ValRMSE']:>10.4f} {row['Gap']:>8.4f}  {status}")

### 4.5. Overfitting / underfitting diagnostics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OVERFITTING / UNDERFITTING DIAGNOSTICS
# Run AFTER Cells 14 (SVR), 16 (RF), and 18 (XGB) have been executed.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

TARGETS = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']
TARGET_LABELS = ['GU EC50', 'GU Emax', 'cAMP EC50', 'cAMP Emax']

# ─────────────────────────────────────────────────────────────────────────────
# 1. TRAIN vs. VALIDATION RMSE  (bias-variance gap)
# ─────────────────────────────────────────────────────────────────────────────

def compute_train_val_rmse(fold_models, model_type, X, y):
    """
    Returns two dicts keyed by target: lists of (train_rmse, val_rmse) per fold.
    model_type: 'svr' | 'rf' | 'xgb'
    """
    train_rmses = {t: [] for t in TARGETS}
    val_rmses   = {t: [] for t in TARGETS}

    for entry in fold_models:
        if model_type == 'svr':
            scaler, model, tr_idx, val_idx = entry
            X_tr  = scaler.transform(X[tr_idx])
            X_val = scaler.transform(X[val_idx])
        else:
            model, tr_idx, val_idx = entry
            X_tr  = X[tr_idx]
            X_val = X[val_idx]

        y_tr_pred  = model.predict(X_tr)
        y_val_pred = model.predict(X_val)

        for i, t in enumerate(TARGETS):
            train_rmses[t].append(np.sqrt(mean_squared_error(y[tr_idx, i],  y_tr_pred[:, i])))
            val_rmses[t].append(  np.sqrt(mean_squared_error(y[val_idx, i], y_val_pred[:, i])))

    return train_rmses, val_rmses


svr_tr_rmse, svr_val_rmse  = compute_train_val_rmse(svr_fold_models,  'svr', X_filtered, y)
rf_tr_rmse,  rf_val_rmse   = compute_train_val_rmse(rf_fold_models,   'rf',  X_filtered, y)
xgb_tr_rmse, xgb_val_rmse  = compute_train_val_rmse(xgb_fold_models,  'xgb', X_filtered, y)

model_results = {
    'SVR':           (svr_tr_rmse,  svr_val_rmse),
    'Random Forest': (rf_tr_rmse,   rf_val_rmse),
    'XGBoost':       (xgb_tr_rmse,  xgb_val_rmse),
}

folds = np.arange(1, 6)

# 4 rows (one per target) × 3 cols (one per model)
fig, axes = plt.subplots(4, 3, figsize=(17, 18), sharey='row')
fig.suptitle('Train vs. Validation RMSE per Fold\n'
             '(Large gap = overfitting  |  Both high = underfitting)',
             fontsize=13, fontweight='bold')

for col, (mname, (tr_rmse, val_rmse)) in enumerate(model_results.items()):
    for row, (t, tlabel) in enumerate(zip(TARGETS, TARGET_LABELS)):
        ax = axes[row, col]
        ax.plot(folds, tr_rmse[t],  'o-',  color='steelblue', label='Train RMSE', lw=2)
        ax.plot(folds, val_rmse[t], 's--', color='tomato',    label='Val RMSE',   lw=2)
        ax.fill_between(folds, tr_rmse[t], val_rmse[t], alpha=0.15, color='orange',
                        label='Gap (overfit zone)')
        ax.set_title(f'{mname}  [{tlabel}]')
        ax.set_xlabel('Fold')
        ax.set_ylabel('RMSE')
        ax.legend(fontsize=8)
        ax.set_xticks(folds)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 2. SUMMARY TABLE  — mean train RMSE vs. mean val RMSE + gap ratio
# ─────────────────────────────────────────────────────────────────────────────

rows = []
for mname, (tr_rmse, val_rmse) in model_results.items():
    for ti, t in enumerate(TARGETS):
        tr_mean  = np.mean(tr_rmse[t])
        val_mean = np.mean(val_rmse[t])
        gap      = val_mean - tr_mean
        ratio    = val_mean / tr_mean if tr_mean > 0 else np.nan

        if ratio > 2.0:
            diagnosis = '⚠️  Overfitting'
        elif val_mean > 0.5 * y[:, ti].std() * 2:
            diagnosis = '⚠️  Underfitting'
        else:
            diagnosis = '✅  Good fit'

        rows.append({
            'Model':           mname,
            'Target':          t,
            'Train RMSE':      round(tr_mean,  4),
            'Val RMSE':        round(val_mean, 4),
            'Gap (Val-Train)': round(gap,       4),
            'Val/Train ratio': round(ratio,     2),
            'Diagnosis':       diagnosis,
        })

summary_df = pd.DataFrame(rows)
print('\n── Overfitting / Underfitting Summary ──────────────────────────────────')
print(summary_df.to_string(index=False))


# ─────────────────────────────────────────────────────────────────────────────
# 3. PREDICTED vs. ACTUAL  (best fold of each model)
# ─────────────────────────────────────────────────────────────────────────────

def get_best_val_preds(fold_models, model_type, fold_metrics, X, y):
    best = np.argmin([
        sum(m[t]['RMSE'] for t in TARGETS) / len(TARGETS)
        for m in fold_metrics
    ])
    entry = fold_models[best]
    if model_type == 'svr':
        scaler, model, tr_idx, val_idx = entry
        X_val = scaler.transform(X[val_idx])
    else:
        model, tr_idx, val_idx = entry
        X_val = X[val_idx]
    return y[val_idx], model.predict(X_val)

svr_yv,  svr_yp  = get_best_val_preds(svr_fold_models,  'svr', svr_fold_metrics,  X_filtered, y)
rf_yv,   rf_yp   = get_best_val_preds(rf_fold_models,   'rf',  rf_fold_metrics,   X_filtered, y)
xgb_yv,  xgb_yp  = get_best_val_preds(xgb_fold_models,  'xgb', xgb_fold_metrics,  X_filtered, y)

model_preds = [
    ('SVR',           svr_yv, svr_yp),
    ('Random Forest', rf_yv,  rf_yp),
    ('XGBoost',       xgb_yv, xgb_yp),
]

# 4 rows × 3 cols
fig, axes = plt.subplots(4, 3, figsize=(16, 18))
fig.suptitle('Predicted vs. Actual  (Best Fold)\n'
             'Points on the diagonal = perfect prediction',
             fontsize=13, fontweight='bold')

for col, (mname, yv, yp) in enumerate(model_preds):
    for row, (ti, tlabel) in enumerate(zip(range(4), TARGET_LABELS)):
        ax = axes[row, col]
        ax.scatter(yv[:, ti], yp[:, ti], alpha=0.6, edgecolors='k',
                   linewidths=0.3, color='steelblue', s=40)
        lo = min(yv[:, ti].min(), yp[:, ti].min())
        hi = max(yv[:, ti].max(), yp[:, ti].max())
        ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Ideal')
        r2 = r2_score(yv[:, ti], yp[:, ti])
        ax.set_title(f'{mname}  [{tlabel}]  R²={r2:.3f}')
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 4. RESIDUAL PLOTS  (best fold)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
fig.suptitle('Residuals vs. Predicted  (Best Fold)\n'
             'Patterns = model misfit  |  Funnel shape = heteroscedasticity',
             fontsize=13, fontweight='bold')

for col, (mname, yv, yp) in enumerate(model_preds):
    for row, (ti, tlabel) in enumerate(zip(range(4), TARGET_LABELS)):
        ax = axes[row, col]
        residuals = yv[:, ti] - yp[:, ti]
        ax.scatter(yp[:, ti], residuals, alpha=0.6, edgecolors='k',
                   linewidths=0.3, color='darkorange', s=40)
        ax.axhline(0, color='red', linestyle='--', lw=1.5)
        ax.set_title(f'{mname}  [{tlabel}]')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Residual (Actual − Predicted)')

plt.tight_layout()
plt.show()

print('\nDiagnostic complete.')
print('Interpretation guide:')
print('  Train/Val RMSE gap  >  2×  →  likely overfitting')
print('  Both train & val RMSE high  →  likely underfitting')
print('  Predicted vs Actual: points hugging the diagonal  →  good fit')
print('  Residuals: no pattern around zero  →  unbiased model')

## 5. Helper — Bias Score for High GU + Low cAMP

For each feature $f$, we define a **4-target bias score**:

$$\\text{BiasScore}(f) = -\\phi_{gu\_ec50}(f) + \\phi_{gu\_emax}(f) - \\phi_{camp\_ec50}(f) + \\phi_{camp\_emax}(f)$$

where $\\phi$ are SHAP values. The sign convention reflects that lower EC50 and higher Emax are desirable. A positive bias score means the feature promotes the desired potency/efficacy profile.

> You can adjust `BIAS_SIGNS` below to change which direction is 'good' for each target.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Desired direction per target
BIAS_SIGNS = {
    'gu_ec50':   -1,  # lower better
    'gu_emax':   +1,  # higher better
    'camp_ec50': +1,  # higher better
    'camp_emax': -1,  # lower better
}

# Optional biological weighting
TARGET_WEIGHTS = {
    'gu_ec50':   0.5,
    'gu_emax':   0.5,
    'camp_ec50': 0.5,
    'camp_emax': 0.5,
}


def normalize(v):
    """Min-max normalize vector to [-1,1] preserving sign"""
    vmin, vmax = v.min(), v.max()
    if vmax == vmin:
        return np.zeros_like(v)
    return 2 * (v - vmin) / (vmax - vmin) - 1


def compute_bias(shap_dict, normalize_targets=True):
    """
    Computes pathway bias score.

    Steps:
    1. Mean SHAP per feature per target
    2. Normalize per target (optional)
    3. Apply optimization direction
    4. Aggregate glucose vs cAMP contributions

    Returns
    -------
    bias : (F,) vector
    """

    mean_shap = {t: sv.mean(axis=0) for t, sv in shap_dict.items()}

    if normalize_targets:
        mean_shap = {t: normalize(v) for t, v in mean_shap.items()}

    # Apply desired direction
    adjusted = {
        t: BIAS_SIGNS[t] * TARGET_WEIGHTS[t] * mean_shap[t]
        for t in mean_shap
    }

    # Glucose pathway
    gu_contrib = (
        adjusted['gu_ec50'] +
        adjusted['gu_emax']
    )

    # cAMP pathway
    camp_contrib = (
        adjusted['camp_ec50'] +
        adjusted['camp_emax']
    )

    # Final bias
    bias = gu_contrib - camp_contrib

    return bias


def bias_score_plot(shap_dict, feature_names, top_n=20, title='Glucose vs cAMP Bias Score'):

    bias = compute_bias(shap_dict)

    idx_sorted = np.argsort(np.abs(bias))[::-1][:top_n]

    top_names = [feature_names[i] for i in idx_sorted]
    top_bias = bias[idx_sorted]

    colors = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]

    plt.figure(figsize=(10, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Bias Score (↑ favors glucose over cAMP)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    rows = {
        'Feature': top_names,
        'BiasScore': top_bias
    }

    for t in shap_dict.keys():
        rows[f'MeanSHAP_{t}'] = shap_dict[t].mean(axis=0)[idx_sorted]

    return pd.DataFrame(rows)


print("Improved pharmacological bias-score helper defined.")

## 6. Explainable AI methods
### 6.1 KernelSHAP — SVR

KernelSHAP is model-agnostic and the appropriate SHAP method for SVR.

In [ ]:
# Use best fold (lowest mean RMSE across all targets)
best_fold_svr = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in svr_fold_metrics
])
scaler_svr, mo_svr_best, tr_idx, val_idx = svr_fold_models[best_fold_svr]

X_tr_sc_svr  = scaler_svr.transform(X_filtered[tr_idx])
X_val_sc_svr = scaler_svr.transform(X_filtered[val_idx])

# KernelSHAP uses a background summary (k-means to keep it tractable)
background = shap.kmeans(X_tr_sc_svr, 30)

shap_svr = {}  # {target: (N, F) array}

for i, (estimator, target) in enumerate(zip(mo_svr_best.estimators_, TARGET_NAMES)):
    print(f'  Computing KernelSHAP for SVR output {i+1}/{len(TARGET_NAMES)}: {target} ...')
    explainer = shap.KernelExplainer(estimator.predict, background)
    sv = explainer.shap_values(X_val_sc_svr, nsamples=200)
    shap_svr[target] = np.array(sv)

# ── Individual SHAP summary plots ─────────────────────────────────────────
for target in TARGET_NAMES:
    shap.summary_plot(shap_svr[target], X_val_sc_svr, feature_names=names_filtered,
                      max_display=20, plot_type='bar', show=True)

# ── Bias score ─────────────────────────────────────────────────────────────
print('\n[SVR] Bias Score — Desired Activity Profile:')
svr_bias_df = bias_score_plot(
    shap_svr, names_filtered, top_n=20,
    title='SVR — Bias Score (Desired Activity Profile)'
)
display(svr_bias_df.head(10))

### 6.2 TreeSHAP + Feature Importance — Random Forest

In [ ]:
best_fold_rf = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in rf_fold_metrics
])
rf_best, tr_idx_rf, val_idx_rf = rf_fold_models[best_fold_rf]

X_tr_rf  = X_filtered[tr_idx_rf]
X_val_rf = X_filtered[val_idx_rf]

# ── (a) MDI Feature Importance ─────────────────────────────────────────────
n_feats = X_tr_rf.shape[1]
fi_per_output = np.zeros((len(TARGET_NAMES), n_feats))
for tree in rf_best.estimators_:
    fi_per_output += tree.feature_importances_.reshape(1, n_feats)
fi_per_output /= len(rf_best.estimators_)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (i, target) in zip(axes.flat, enumerate(TARGET_NAMES)):
    top20_idx = np.argsort(fi_per_output[i])[::-1][:20]
    ax.barh([names_filtered[j] for j in top20_idx][::-1],
             fi_per_output[i][top20_idx][::-1], color='teal')
    ax.set_title(f'RF MDI Importance — {target}')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest Feature Importance (MDI)', fontweight='bold')
plt.tight_layout()
plt.show()

# ── (b) TreeSHAP ───────────────────────────────────────────────────────────
rf_explainer = shap.TreeExplainer(rf_best)
shap_rf_vals = rf_explainer.shap_values(X_val_rf)  # list of n_outputs arrays (N, F)

shap_rf = {}
if isinstance(shap_rf_vals, list):
    for i, target in enumerate(TARGET_NAMES):
        shap_rf[target] = np.array(shap_rf_vals[i])
elif shap_rf_vals.ndim == 3:
    for i, target in enumerate(TARGET_NAMES):
        shap_rf[target] = shap_rf_vals[:, :, i]
else:
    raise ValueError('Unexpected SHAP output shape from RF TreeExplainer')

print('\n[RF] Bias Score — Desired Activity Profile:')
rf_bias_df = bias_score_plot(
    shap_rf, names_filtered, top_n=20,
    title='Random Forest — Bias Score (Desired Activity Profile)'
)
display(rf_bias_df.head(10))

### 6.3 TreeSHAP + Feature Importance — XGBoost

In [ ]:
best_fold_xgb = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in xgb_fold_metrics
])
xgb_best, tr_idx_xgb, val_idx_xgb = xgb_fold_models[best_fold_xgb]

X_tr_xgb  = X_filtered[tr_idx_xgb]
X_val_xgb = X_filtered[val_idx_xgb]

# ── (a) Built-in feature importance ───────────────────────────────────────
# NOTE: 'gain' importance is not supported for multi_output_tree in XGBoost.
# We use 'weight' (split frequency) instead, which is supported.
fi_xgb = xgb_best.get_booster().get_score(importance_type='weight')

# get_score() returns a dict {feature_name: score} and may omit zero-importance
# features — remap to a full-length array aligned with maccs_names_filtered
n_feats = X_tr_xgb.shape[1]
fi_xgb_arr = np.zeros(n_feats)
for fname, score in fi_xgb.items():
    # XGBoost uses 'f0', 'f1', ... when no feature names are set
    idx = int(fname.replace('f', ''))
    fi_xgb_arr[idx] = score

top20_xgb = np.argsort(fi_xgb_arr)[::-1][:10]
plt.figure(figsize=(10, 6))
plt.barh([names_filtered[i] for i in top20_xgb][::-1],
          fi_xgb_arr[top20_xgb][::-1], color='darkorange')
plt.xlabel('Weight Importance (split count)')
plt.title('XGBoost Feature Importance (Multi-Output, weight)')
plt.tight_layout()
plt.show()

# ── (b) KernelSHAP per output ──────────────────────────────────────────────
# NOTE: shap.TreeExplainer does not support XGBoost's multi_output_tree
# (vector-leaf trees) — raises "vector-leaf is not yet supported".
# KernelSHAP is model-agnostic and works correctly here.
# We use a k-means background summary to keep it tractable.

print('Computing KernelSHAP for XGBoost (this may take a few minutes)...')
bg_xgb = shap.kmeans(X_tr_xgb, 30)

shap_xgb = {}
for i, target in enumerate(TARGET_NAMES):
    print(f'  Output {i+1}/{len(TARGET_NAMES)}: {target} ...')
    def predict_output(x, idx=i):
        return xgb_best.predict(x)[:, idx]
    ex = shap.KernelExplainer(predict_output, bg_xgb)
    shap_xgb[target] = ex.shap_values(X_val_xgb, nsamples=200)

print('\n[XGB] Bias Score — Desired Activity Profile:')
xgb_bias_df = bias_score_plot(
    shap_xgb, names_filtered, top_n=20,
    title='XGBoost — Bias Score (Desired Activity Profile)'
)
display(xgb_bias_df.head(10))

## 7. Consensus Bias Score Across Models

Aggregate MACCS-level bias scores from SVR, RF, and XGBoost to find features that **all models agree** drive high GU + low cAMP.

In [ ]:
def get_bias_vector(shap_dict):
    """Returns (F,) bias vector from shap_dict."""
    return compute_bias(shap_dict)

bias_svr  = get_bias_vector(shap_svr)
bias_rf   = get_bias_vector(shap_rf)
bias_xgb  = get_bias_vector(shap_xgb)

# Normalise each to [-1, 1] before averaging
def normalise(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

consensus_bias = (normalise(bias_svr) + normalise(bias_rf) + normalise(bias_xgb)) / 3.0

top_n = 25
idx_top = np.argsort(np.abs(consensus_bias))[::-1][:top_n]
names_top = [names_filtered[i] for i in idx_top]
vals_top  = consensus_bias[idx_top]

colors = ['#1976D2' if v > 0 else '#D32F2F' for v in vals_top]
plt.figure(figsize=(11, 7))
plt.barh(names_top[::-1], vals_top[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Normalised Consensus Bias Score')
plt.title('Consensus Bias Score (SVR + RF + XGBoost)\n↑ = Desired Activity Profile (Low EC50 & High Emax)', fontsize=13)
plt.tight_layout()
plt.show()

consensus_df = pd.DataFrame({
    'Feature':        names_top,
    'ConsensusScore': vals_top,
    'SVR_bias':  bias_svr[idx_top],
    'RF_bias':   bias_rf[idx_top],
    'XGB_bias':  bias_xgb[idx_top]
}).sort_values('ConsensusScore', ascending=False)

print('Top features driving desired activity profile (positive score):')
display(consensus_df[consensus_df['ConsensusScore'] > 0].head(10))
print('\nTop features that hinder the desired outcome (negative score):')
display(consensus_df[consensus_df['ConsensusScore'] < 0].head(10))

## 8. Morgan Bit Interpretation Helper

Morgan fingerprint bits are hashed circular substructures and do not have fixed chemical definitions (unlike MACCS keys). The recommended workflow is:

1. **Identify the top-scoring bit indices** from the consensus bias DataFrame.
2. **Use `AllChem.GetMorganBitInfo()`** on a representative molecule to see which atom/radius generated each bit.
3. **Visualise fragments** with `Draw.DrawMorganBit()` for a pictorial interpretation.

The cell below implements steps 1–3 automatically for the top positive-bias bits.

In [ ]:
# ── Morgan bit fragment visualisation ──────────────────────────────────────
# For each top-bias bit, find which molecule best exemplifies it and draw
# the circular substructure it encodes.

TOP_N_BITS = 10

# Extract positive-bias top bits from consensus_df
pos_bits = consensus_df[consensus_df['ConsensusScore'] > 0].head(TOP_N_BITS)
top_bit_indices = [int(f.split('_')[1]) for f in pos_bits['Feature']]

print('Top positive-bias Morgan bits and their bit indices:')
for feat, score in zip(pos_bits['Feature'], pos_bits['ConsensusScore']):
    print(f'  {feat}  →  Consensus bias score = {score:.4f}')

# For each bit, find a molecule that has it set and render the fragment
mols_for_bits = []
legends = []

for bit_idx in top_bit_indices:
    # Search dataset for a molecule that has this bit set
    for smi in df['SMILES']:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        bi = {}  # bit_info dict
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS, bitInfo=bi
        )
        if bit_idx in bi:
            mols_for_bits.append((mol, bit_idx, bi))
            legends.append(f'Bit {bit_idx}')
            break

if mols_for_bits:
    imgs = []
    for mol, bit_idx, bi in mols_for_bits:
        img = Draw.DrawMorganBit(mol, bit_idx, bi, molSize=(250, 250))
        imgs.append(img)

    # Display as a grid
    from PIL import Image
    import math
    ncols = min(5, len(imgs))
    nrows = math.ceil(len(imgs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten()
    for ax, img, leg in zip(axes, imgs, legends):
        ax.imshow(img)
        ax.set_title(leg, fontsize=10)
        ax.axis('off')
    for ax in axes[len(imgs):]:
        ax.axis('off')
    plt.suptitle('Top Positive-Bias Morgan Fragments', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No matching molecules found for the top bits in this dataset.')

consensus_df['BitIndex'] = consensus_df['Feature'].apply(lambda x: int(x.split('_')[1]))
print('\nFull top-feature table with bit indices:')
display(consensus_df[['Feature', 'BitIndex', 'ConsensusScore', 'SVR_bias', 'RF_bias', 'XGB_bias']].head(TOP_N_BITS))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# XGBoost → Top Morgan Bits → Fragment Visualisation (ROBUST VERSION)
# ─────────────────────────────────────────────────────────────────────────────

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd

TOP_N_BITS = 10
MAX_MOLS_PER_BIT = 1   # increase to 2–3 if you want more examples per bit

# ── STEP 1: Get top XGBoost features ─────────────────────────────────────────
top_feat_idx = np.argsort(fi_xgb_arr)[::-1][:TOP_N_BITS]

print('Top XGBoost features:')
for i in top_feat_idx:
    print(f'  {names_filtered[i]} → importance = {fi_xgb_arr[i]:.4f}')


# ── STEP 2: Extract valid Morgan bit indices ─────────────────────────────────
top_bit_indices = []
bit_to_feature = {}

for i in top_feat_idx:
    fname = names_filtered[i]

    # Only keep Morgan bits (expects "Bit_123")
    if isinstance(fname, str) and fname.startswith('Bit_'):
        try:
            bit_idx = int(fname.split('_')[1])
            top_bit_indices.append(bit_idx)
            bit_to_feature[bit_idx] = fname
        except:
            print(f'⚠️ Could not parse bit index from {fname}')
    else:
        print(f'⚠️ Skipping non-Morgan feature: {fname}')

print('\nValid Morgan bits selected:')
for b in top_bit_indices:
    print(f'  {bit_to_feature[b]} → bit {b}')

if len(top_bit_indices) == 0:
    raise ValueError("❌ No Morgan bits found among top features. Check feature naming.")


# ── STEP 3: Validate fingerprint parameters ──────────────────────────────────
print(f'\nFingerprint settings:')
print(f'  Radius: {MORGAN_RADIUS}')
print(f'  nBits : {MORGAN_NBITS}')


# ── STEP 4: Find molecules containing each bit ───────────────────────────────
mols_for_bits = []
legends = []

for bit_idx in top_bit_indices:
    found_count = 0

    for smi in df['SMILES']:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue

        bi = {}
        _ = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=MORGAN_RADIUS,
            nBits=MORGAN_NBITS,
            bitInfo=bi
        )

        if bit_idx in bi:
            mols_for_bits.append((mol, bit_idx, bi))
            legends.append(f'{bit_to_feature[bit_idx]}')
            found_count += 1

            if found_count >= MAX_MOLS_PER_BIT:
                break

    if found_count == 0:
        print(f'⚠️ Bit {bit_idx} not found in dataset')


# ── STEP 5: Draw fragments ───────────────────────────────────────────────────
if mols_for_bits:
    imgs = []

    for mol, bit_idx, bi in mols_for_bits:
        try:
            img = Draw.DrawMorganBit(
                mol,
                bit_idx,
                bi,
                molSize=(250, 250)
            )
            imgs.append(img)
        except Exception as e:
            print(f'⚠️ Failed to draw bit {bit_idx}: {e}')

    if len(imgs) == 0:
        print('❌ No images could be generated.')
    else:
        ncols = min(5, len(imgs))
        nrows = math.ceil(len(imgs) / ncols)

        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
        axes = np.array(axes).flatten()

        for ax, img, leg in zip(axes, imgs, legends):
            ax.imshow(img)
            ax.set_title(leg, fontsize=9)
            ax.axis('off')

        # Turn off unused axes
        for ax in axes[len(imgs):]:
            ax.axis('off')

        plt.suptitle(
            'Top XGBoost Morgan Fragments',
            fontsize=13,
            fontweight='bold'
        )
        plt.tight_layout()
        plt.show()

else:
    print('❌ No matching molecules found for any selected bits.')


# ── STEP 6: Summary table ────────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Feature': [bit_to_feature[b] for b in top_bit_indices],
    'BitIndex': top_bit_indices,
    'Importance': [fi_xgb_arr[names_filtered.index(bit_to_feature[b])]
                   for b in top_bit_indices]
})

print('\nTop Morgan feature summary:')
display(summary_df)